# Scatter Plot of 3 Different Worm Trajectories

## Imports
Load all required libraries for file handling, data manipulation, and plotting.

In [ ]:
import h5py
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize
import numpy as np

## File Paths
Define the file paths for the three conditioning conditions: mock, aversive, and sexual.
Each path points to a HDF5 file containing trajectory data for a single worm.

In [ ]:
filepaths = {
    "mock conditioned": "Chemotaxis-Data-and-Analysis/Mock_worms/chemotaxis_mock_210825_3_20250722_161220/metadata_featuresN_oneworm.hdf5",
    "aversively conditioned": "Chemotaxis-Data-and-Analysis/Aversive_worms/chemotaxis_avsv_24_1_23_01_20240124_140022/metadata_featuresN_oneworm.hdf5",
    "sexually conditioned": "Chemotaxis-Data-and-Analysis/sexually_conditioned_worms/chemotaxis_sexc_24_1_26_09_20240126_143858/metadata_featuresN_oneworm.hdf5",
}

## File Validation
Check that all three HDF5 files exist at the expected paths before attempting to load them.
If any file is missing, the script will print a helpful message and stop.


In [ ]:
for condition, filepath in filepaths.items():
    if not os.path.exists(filepath):
        print(f"\nFile not found: {filepath}!")
        print("Please ensure the Chemotaxis-Data-and-Analysis repository is cloned in this folder.")
        raise FileNotFoundError(f"Missing file for condition: {condition}")

## Parameters
Set the scale factor and colour assignments for each condition.

- **Scale factor:** Tierpsy stores coordinates in pixels; multiplying by 13 converts them to micrometres.
- **Colours:** Each conditioning type is assigned a distinct colour for the scatter plot.

In [ ]:
# Scale factor to convert pixel coordinates to micrometers
scale_factor = 13

# Colors for each condition
colors = {
    "mock conditioned": "red",
    "aversively conditioned": "blue",
    "sexually conditioned": "green",
}

## Load Data and Plot Trajectories
For each condition:
1. Open the HDF5 file and read the `trajectories_data` table into a Pandas DataFrame.
2. Extract the `coord_x` and `coord_y` columns and convert from pixels to micrometres.
3. Normalise coordinates so each trajectory starts at the origin `(0, 0)` — this allows meaningful visual comparison across animals regardless of where they were placed on the plate.
4. Plot the trajectory as a scatter plot with small point size and partial transparency.

In [ ]:
# Create figure and axis
plt.figure(figsize=(6, 6))

for condition, filepath in filepaths.items():
    print(f"Loading {condition} dataset...")

    with h5py.File(filepath, "r") as f:
        traj_data = pd.DataFrame(f["trajectories_data"][:])

    # Get x and y coordinates and convert to micrometers
    x_coords = np.array(traj_data["coord_x"]) * scale_factor
    y_coords = np.array(traj_data["coord_y"]) * scale_factor

    # Normalize so first position is at (0, 0)
    x_normalized = x_coords - x_coords[0]
    y_normalized = y_coords - y_coords[0]

    # Plot the trajectory as a scatter
    plt.scatter(
        x_normalized,
        y_normalized,
        s=2,
        c=colors[condition],
        label=condition,
        alpha=0.6,
    )

    print(f"  Plotted {len(x_normalized)} points")

## Formatting and Odour Gradient Colourbar
Add axis labels, a title, legend, and a decorative colourbar indicating the conceptual odour gradient direction.
The colourbar is illustrative — it does not represent measured concentration values.

In [ ]:
# Labels and formatting
plt.xlabel("X Position (micrometers)", fontsize=12)
plt.ylabel("Y Position (micrometers)", fontsize=12)
plt.title("Worm Chemotaxis Trajectories", fontsize=14, fontweight="bold")
plt.legend(fontsize=11, loc="best")
plt.grid(True, alpha=0.3)
plt.axis("equal")

# Add odor gradient colorbar
sm = cm.ScalarMappable(cmap=cm.Blues, norm=Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=plt.gca(), orientation='horizontal', pad=0.15, shrink=0.8)
cbar.set_label('Odor Gradient', fontsize=11)
cbar.ax.set_xticklabels(['Weak', '', '', '', 'Strong'])

## Save and Display
Save the figure as a high-resolution PNG to the `images/` folder, then display it inline.

In [ ]:
output_path = "images/worm_trajectories_comparison.png"
os.makedirs("images", exist_ok=True)
plt.savefig(output_path, dpi=300, bbox_inches="tight")
print(f"\nPlot saved to: {output_path}")

plt.show()